# 01 · Exploración del dataset GitHub Events (GH Archive)

Fuente: https://www.gharchive.org/ — un archivo `YYYY-MM-DD-H.json.gz` por hora con todos los eventos públicos de GitHub.

Objetivo de este notebook: documentar **origen, estructura y características** del dataset (requisito 1 del proyecto).

In [ ]:
import sys, os
sys.path.insert(0, "../src")          # modulos del proyecto
import config
from datetime import datetime, timezone, date
from pprint import pprint

In [ ]:
from extract import hour_to_process, download_hour, iter_events
hour = hour_to_process()          # hora utc mas reciente publicada
path = download_hour(hour)
print(hour, '->', path, f'{path.stat().st_size/1e6:.1f} MB')

In [ ]:
import itertools, collections
sample = list(itertools.islice(iter_events(path), 20000))
print('eventos en la muestra:', len(sample))
pprint(sample[0])

In [ ]:
# tipos de evento y su frecuencia
types = collections.Counter(e['type'] for e in sample)
for t, n in types.most_common(): print(f'{t:<30}{n:>7}  {n/len(sample):6.1%}')

In [ ]:
# el payload cambia segun el tipo
for t in ['PushEvent', 'IssuesEvent', 'WatchEvent', 'PullRequestEvent']:
    ev = next((e for e in sample if e['type'] == t), None)
    if ev: print(t, '-> claves de payload:', sorted(ev['payload'].keys()))

In [ ]:
# conteo total del archivo
total = sum(1 for _ in iter_events(path))
print('eventos en la hora completa:', total)

In [ ]:
# cuantos pasan la validacion
from clean import clean_events
clean, stats = clean_events(iter_events(path), max_events=10**9)
pprint(stats)